   # Insurance Claims Analytics & Fraud Risk Analysis
   Analysing a synthetic motor insurance claims dataset to understand claims volume, processing performance (SLA), and patterns associated with potentially fraudulent claims.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/insurance_claims.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df["region"].value_counts()

In [ ]:
df["channel"].value_counts()

In [ ]:
df["claim_type"].value_counts()

In [ ]:
(pd.to_datetime(df["claim_date"]) < pd.to_datetime(df["incident_date"])).sum()

## Data quality issues found

## Data quality issues found

The raw dataset contains 15,140 rows and 24 columns. Exploration revealed the following issues:

1. **Duplicate rows:** 140 exact duplicate records.
2. **Missing customer age:** 222 records have no age recorded.
3. **Impossible customer ages:** 6 records have ages outside a realistic range (e.g. 0, -3, 150, 212).
4. **Negative claim amounts:** 5 claims have a negative value, which is not possible and suggests data entry errors.
5. **Inconsistent region labels:** 233 records use non-standard formatting: 152 in lowercase (e.g. "north") and 81 with extra spaces (e.g. " North ").
6. **Inconsistent channel labels:** 61 records use "mobile app" instead of "Mobile App".
7. **Inconsistent claim type labels:** 40 records use "Third Party" instead of "Third-party".
8. **Invalid date logic:** 12 claims have a claim date earlier than the incident date, which is not possible.

**Other observations**
- `claim_date` and `incident_date` are stored as text and need converting to dates.
- `customer_satisfaction` has 1,578 missing values. These represent customers who did not complete the satisfaction survey, so they will be **kept as missing** rather than filled in, to avoid biasing the satisfaction analysis.

In [ ]:
print("Rows before:", len(df))
df = df.drop_duplicates()
print("Rows after:", len(df))

### Fix 2: Standardise text labels
Inconsistent casing, extra spaces and naming variations would split one category into several in charts and SQL GROUP BYs.


In [ ]:
df["region"] = df["region"].str.strip().str.title()
df["channel"] = df["channel"].str.strip().str.title()
df["claim_type"] = df["claim_type"].str.strip().replace({"Third Party": "Third-party"})

print("Regions:", df["region"].nunique())
print("Channels:", df["channel"].nunique())
print("Claim types:", df["claim_type"].nunique())

### Fix 3: Convert dates and remove impossible records
Dates are converted from text to datetime. Claims dated before the incident are logically impossible, so they are removed.

In [ ]:
df["claim_date"] = pd.to_datetime(df["claim_date"])
df["incident_date"] = pd.to_datetime(df["incident_date"])

bad_dates = df["claim_date"] < df["incident_date"]
print("Claims before incident:", bad_dates.sum())
df = df[~bad_dates]

### Fix 4: Handle invalid and missing ages
Ages below 18 or above 100 are treated as invalid and set to missing. All missing ages are then filled with the median age, which is robust to outliers.

In [ ]:
df.loc[(df["customer_age"] < 18) | (df["customer_age"] > 100), "customer_age"] = np.nan
print("Missing ages to fill:", df["customer_age"].isnull().sum())

median_age = df["customer_age"].median()
df["customer_age"] = df["customer_age"].fillna(median_age)
print("Median age used:", median_age)

### Fix 5: Remove negative claim amounts
A claim amount cannot be negative. These are treated as data entry errors and removed.

In [ ]:
print("Negative amounts:", (df["claim_amount"] <= 0).sum())
df = df[df["claim_amount"] > 0]
print("Final rows:", len(df))

In [ ]:
df.info()

## Feature engineering
New business fields are created to support SLA, settlement, documentation and risk analysis. The company's processing target (SLA) is assumed to be 7 days.

In [ ]:
df["settlement_ratio"] = df["settlement_amount"] / df["claim_amount"]
df["sla_status"] = np.where(df["processing_days"] <= 7, "Within SLA", "Outside SLA")
df["doc_status"] = np.where(df["documents_submitted"] >= df["documents_required"], "Complete", "Incomplete")
df["value_band"] = np.where(df["claim_amount"] >= 200000, "High Value", "Standard")
df["reporting_lag_days"] = (df["claim_date"] - df["incident_date"]).dt.days
df["claim_month"] = df["claim_date"].dt.to_period("M").astype(str)

df["sla_status"].value_counts()

In [ ]:
df["previous_claims"].value_counts().sort_index()

Only ~22 customers have 5+ previous claims, so a threshold of 3+ is used in the risk score to capture a meaningful group of repeat claimants.

### Fraud-risk score
An exploratory, rule-based risk score combining several claim characteristics. This is not a production fraud model; it is used to identify patterns associated with potentially suspicious claims.

| Rule | Points |
|---|---|
| Claim amount ≥ ₹200,000 | +2 |
| 3+ previous claims | +2 |
| Investigation required | +2 |
| Processing > 14 days | +1 |
| Incomplete documents | +1 |
| High/Critical severity | +1 |

Risk level: 0–2 = Low, 3–5 = Medium, 6+ = High

In [ ]:
df["risk_score"] = (
    np.where(df["claim_amount"] >= 200000, 2, 0)
    + np.where(df["previous_claims"] >= 3, 2, 0)
    + np.where(df["investigation_required"] == 1, 2, 0)
    + np.where(df["processing_days"] > 14, 1, 0)
    + np.where(df["doc_status"] == "Incomplete", 1, 0)
    + np.where(df["severity"].isin(["High", "Critical"]), 1, 0)
)

df["risk_level"] = pd.cut(df["risk_score"], bins=[-1, 2, 5, 9],
                          labels=["Low", "Medium", "High"])
df["risk_level"].value_counts()

In [ ]:
summary = df.groupby("risk_level", observed=True).agg(
    claims=("claim_id", "count"),
    fraud_rate=("fraud_flag", "mean"),
    total_value=("claim_amount", "sum"),
)
summary["share_of_claims"] = summary["claims"] / summary["claims"].sum()
summary["share_of_value"] = summary["total_value"] / summary["total_value"].sum()
summary.round(3)

**Insight:** High-risk claims make up only ~4.4% of claim volume but ~18% of total claim value, and their fraud rate (~55%) is roughly 50x that of low-risk claims. This suggests the rule-based score is effective at prioritising claims for investigation.

**Insight:** High-risk claims make up only ~4.4% of claim volume but ~18% of total claim value, and their fraud rate (~55%) is roughly 50x that of low-risk claims. This suggests the rule-based score is effective at prioritising claims for investigation.

In [ ]:
df.to_csv("data/insurance_claims_clean.csv", index=False)

## Load cleaned data into SQLite for SQL analysis

In [ ]:
import sqlite3

clean = pd.read_csv("data/insurance_claims_clean.csv")
conn = sqlite3.connect("data/insurance_claims.db")
clean.to_sql("insurance_claims", conn, if_exists="replace", index=False)

print(pd.read_sql("SELECT COUNT(*) AS total_claims FROM insurance_claims", conn))
conn.close()